# Hong-to-YOLO26 full transfer — Faruq-v3

Notebook fail-fast ini menjalankan gate statis terlebih dahulu. Training hanya satu kandidat penuh `HF`, seed 42, validation-only. `D0` memakai checkpoint yang sudah selesai; test tidak tersedia dan tidak dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU sebelum screening.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
BASELINE = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/hong-yolo26-transfer-v1'
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT :', PROJECT_ROOT)
print('DATA    :', DATA_ROOT)
print('D0      :', BASELINE)
print('OUTPUT  :', OUTPUT_ROOT)

## Gate 1 — audit arsitektur tanpa training

Sel ini wajib PASS sebelum GPU training. Ia menguji transfer pretrained, forward batch 1/2, backward, state reload, output dual-branch, dan memastikan DSConv tidak hilang saat Ultralytics melakukan fusion.

In [ ]:
import json
from coffee_detector.hong_transfer.audit import static_architecture_audit

STATIC_PATH = OUTPUT_ROOT / 'val_reports/HF_seed42_architecture_audit.json'
static = static_architecture_audit(
    REPO / 'configs/coffee_fg/models/yolo26n-p3.yaml',
    STATIC_PATH,
    nc=21,
    weights=REPO / 'yolo26n.pt',
    image_size=128,
)
print(json.dumps({
    'static_gate': static['static_gate'],
    'module_counts': static['module_counts'],
    'parameters': static['parameters'],
    'finite_gradients': static['finite_gradients'],
    'state_reload_equal': static['state_reload_equal'],
}, indent=2))
print('SAVED:', STATIC_PATH)
assert static['static_gate'] == 'PASS'

## Gate 2 — satu seed full transfer

Setelah hasil Gate 1 dinilai, ubah `RUN_SCREEN=True`. Checkpoint ditulis langsung ke folder proyek Drive setiap epoch oleh Ultralytics; setelah runtime putus, jalankan ulang notebook untuk resume dari `last.pt`.

In [ ]:
RUN_SCREEN = False  # ubah True hanya setelah static gate dinilai

if RUN_SCREEN:
    from coffee_detector.experiments.run_hong_yolo26_transfer import run_hong_yolo26_transfer
    result = run_hong_yolo26_transfer(
        DATA_ROOT,
        GROUPED_SUMMARY,
        BASELINE,
        OUTPUT_ROOT,
        seed=42,
        device='0',
        weights=REPO / 'yolo26n.pt',
    )
    print(json.dumps(result['decision'], indent=2, ensure_ascii=False))
    print('NEXT:', result['next_action'])
    print('SAVED:', result['summary'])
else:
    print('STOP setelah static gate. Kirim ringkasannya sebelum mengaktifkan training.')

In [ ]:
DECISION = OUTPUT_ROOT / 'val_reports/hong_full_transfer_seed42_decision.json'
if DECISION.is_file():
    final = json.loads(DECISION.read_text(encoding='utf-8'))
    print(json.dumps(final['decision'], indent=2, ensure_ascii=False))
    print('TEST OPENED:', final['test_opened'])
    print('NEXT:', final['next_action'])
else:
    print('Decision belum tersedia; training belum dijalankan atau belum selesai.')